# LimiX Regression Example

This notebook demonstrates how to use the FAIM SDK's **TabularClient** with **LimiX** for tabular regression tasks.

LimiX is a foundation model for tabular machine learning that supports both classification and regression.

## Setup

Install dependencies and import required libraries.

In [11]:
!python3 -m pip install scikit-learn

/Users/andreichernov/Documents/Personal/research/FAIM/faim-client/.venv/bin/python3: No module named pip


In [ ]:
import numpy as np
from sklearn.datasets import fetch_california_housing

from faim_sdk import LimiXPredictRequest, TabularClient

## Load and Prepare Data

Load the California housing regression dataset from scikit-learn.

In [ ]:
# Load California housing dataset
house_data = fetch_california_housing()
X, y = house_data.data, house_data.target

# Create a simple 50/50 split for demonstration
split_idx = len(X) // 2
X_train = X[:split_idx].astype(np.float32)
y_train = y[:split_idx].astype(np.float32)
X_test = X[split_idx:].astype(np.float32)
y_test = y[split_idx:].astype(np.float32)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"Number of features: {X_train.shape[1]}")
print(f"Target range: [{y_train.min():.2f}, {y_train.max():.2f}]")

## Initialize TabularClient

Create a client to interact with the LimiX model.

In [ ]:
# Initialize the client
client = TabularClient(
    base_url="https://api.faim.it.com",
    api_key="your-api-key",  # Replace with your actual API key
    timeout=120.0,
)

print("TabularClient initialized!")

## Create Regression Request

Prepare a LimiX regression request.

In [ ]:
# Create a LimiX regression request
request = LimiXPredictRequest(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    task_type="Regression",  # Regression task
    use_retrieval=False,
    model_version="1",
)

print("Request prepared:")
print(f"  X_train shape: {request.X_train.shape}")
print(f"  X_test shape: {request.X_test.shape}")
print(f"  Task type: {request.task_type}")

## Make Predictions

Send the request to LimiX and get regression predictions.

In [ ]:
try:
    # Make predictions
    response = client.predict(request)

    print(f"Predictions shape: {response.predictions.shape}")
    print(f"First 10 predictions: {response.predictions[:10]}")
    print(f"Prediction range: [{response.predictions.min():.2f}, {response.predictions.max():.2f}]")

    print("\nMetadata:")
    for key, value in response.metadata.items():
        print(f"  {key}: {value}")

except Exception as e:
    print(f"Error: {e}")
    print("\nMake sure your API key is valid and the service is available.")

## Evaluate Results

Calculate regression metrics.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    y_pred = response.predictions

    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print("Regression Metrics:")
    print(f"  MSE:  {mse:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE:  {mae:.4f}")
    print(f"  R²:   {r2:.4f}")
except NameError:
    print("Run prediction cell first to evaluate metrics.")

## Residual Analysis

Analyze prediction errors.

In [ ]:
try:
    residuals = y_test - y_pred

    print("Residual Statistics:")
    print(f"  Mean:   {residuals.mean():.4f}")
    print(f"  Std:    {residuals.std():.4f}")
    print(f"  Min:    {residuals.min():.4f}")
    print(f"  Max:    {residuals.max():.4f}")
except NameError:
    print("Run prediction cell first.")

## Using Retrieval-Augmented Inference

Enable RAI for potentially better accuracy.

In [ ]:
# Request with retrieval-augmented inference
request_with_rai = LimiXPredictRequest(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    task_type="Regression",
    use_retrieval=True,  # Enable RAI
    model_version="1",
)

print("Request with RAI enabled prepared.")
# response_rai = client.predict(request_with_rai)

## Async Example

Use async predictions for concurrent operations.

In [ ]:
async def async_regression():
    """Make async regression prediction."""
    request = LimiXPredictRequest(X_train=X_train, y_train=y_train, X_test=X_test, task_type="Regression")

    try:
        response = await client.predict_async(request)
        print(f"Async prediction successful: {response.predictions.shape}")
        return response
    except Exception as e:
        print(f"Async error: {e}")


# Uncomment to run
# response = asyncio.run(async_regression())

## Error Handling

Handle specific exceptions from the SDK.

In [ ]:
from faim_sdk import AuthenticationError, ErrorCode, ValidationError


def make_prediction_with_error_handling():
    try:
        request = LimiXPredictRequest(X_train=X_train, y_train=y_train, X_test=X_test, task_type="Regression")
        response = client.predict(request)
        return response

    except ValidationError as e:
        print(f"Validation Error: {e.message}")
        if e.error_code == ErrorCode.INVALID_SHAPE:
            print("Check array shapes.")
        print(f"Request ID: {e.error_response.request_id}")

    except AuthenticationError as e:
        print(f"Auth Error: {e.message}")
        print("Check your API key.")


# Test error handling
# make_prediction_with_error_handling()